# 🤖 AI Job Analyst V2 — GitHub-Ready Application Layer

**Workflow:** CV PDF → CV text extraction → JD text → skill/experience/education/responsibility analysis → unified score → recommendation → evidence-grounded chatbot.

> **Important:** This notebook is an application/UI layer and depends on the final V2.22 analysis engine. The engine placeholder must be replaced with the tested engine code before the notebook can perform analysis.


In [ ]:
# 1. DEPENDENCIES
# Install once in your environment:
# pip install -r requirements.txt

print("Install dependencies from requirements.txt before running this notebook.")


In [ ]:
# ============================================================
# 2. IMPORTS
# ============================================================

import gradio as gr
from PyPDF2 import PdfReader

print("Imports successful.")


## 3. V2.22 ENGINE

Paste your **final V2.22 engine code** in the next cell if you want this notebook to be completely independent.

It must define at least:

- `run_complete_job_analysis(cv_text_input, jd_text_input)`
- `generate_chat_response_v2(...)`

Your V2.22 engine should internally contain the tested:
- JD parser
- skill aliases/extraction
- skill matching
- professional experience analysis
- education extraction/matching
- responsibility analysis
- unified scoring
- eligibility/recommendation


In [ ]:
# ============================================================
# 3. V2.22 ENGINE PLACEHOLDER
# ============================================================
#
# IMPORTANT:
# Paste your FINAL V2.22 backend/engine code here.
#
# Required functions after running this cell:
#
#   run_complete_job_analysis()
#   generate_chat_response_v2()
#
# If those functions are already defined in the current
# Colab runtime, you can leave this cell unchanged.
#
# ============================================================

print("V2.22 engine cell ready.")
print("Required functions:")
print("  - run_complete_job_analysis")
print("  - generate_chat_response_v2")


In [ ]:
# ============================================================
# 4. VERIFY V2.22 ENGINE
# ============================================================

required_functions = [
    "run_complete_job_analysis",
    "generate_chat_response_v2"
]

missing = [
    name
    for name in required_functions
    if name not in globals()
]

if missing:
    print("⚠️ Missing V2.22 functions:")
    for name in missing:
        print("  -", name)
    print()
    print("Paste/run your final V2.22 engine code in Section 3.")
else:
    print("✅ V2.22 engine detected.")


In [ ]:
# ============================================================
# 5. PDF → TEXT
# ============================================================

def extract_cv_pdf_text(pdf_path):

    if pdf_path is None:
        raise ValueError("Please upload your CV PDF.")

    reader = PdfReader(pdf_path)

    pages = []

    for page in reader.pages:
        try:
            text = page.extract_text()
        except Exception:
            text = ""

        if text:
            pages.append(text)

    cv_text = "\n".join(pages).strip()

    if not cv_text:
        raise ValueError(
            "No readable text was extracted from the CV PDF."
        )

    return cv_text


print("PDF extraction function ready.")


V2 — CV Text Extraction
CV text successfully extracted and normalized.
Candidate profile extraction stage completed.


In [ ]:
# ============================================================
# 6. ANALYSIS CALLBACK
# ============================================================

def analyze_cv_pdf_and_jd_text(cv_pdf, jd_text):

    if cv_pdf is None:
        return (
            "⚠️ Please upload your CV PDF.",
            "",
            "",
            None,
            None
        )

    if not jd_text or not jd_text.strip():
        return (
            "⚠️ Please paste the Job Description.",
            "",
            "",
            None,
            None
        )

    if "run_complete_job_analysis" not in globals():
        return (
            "❌ V2.22 engine is not loaded. Run the V2.22 engine section first.",
            "",
            "",
            None,
            None
        )

    try:
        cv_text_input = extract_cv_pdf_text(cv_pdf)
    except Exception as e:
        return (
            f"❌ CV extraction failed:\n\n{e}",
            "",
            "",
            None,
            None
        )

    try:
        analysis = run_complete_job_analysis(
            cv_text_input=cv_text_input,
            jd_text_input=jd_text
        )
    except Exception as e:
        return (
            f"❌ Analysis failed:\n\n{e}",
            "",
            "",
            None,
            None
        )

    decision = analysis["decision"]

    overall = analysis["overall_score"]

    skill_score = analysis["skill_result"]["skill_score"]

    experience_score = analysis["experience_result"]["experience_match"]

    education_score = analysis["education_result"]["score"]

    responsibility_score = (
        analysis["responsibility_result"]["overall_score"]
    )

    required_matched = analysis["skill_result"]["required_matched"]
    required_missing = analysis["skill_result"]["required_missing"]

    preferred_matched = analysis["skill_result"]["preferred_matched"]
    preferred_missing = analysis["skill_result"]["preferred_missing"]

    summary = f"""
# 📊 Overall Match: **{overall}%**

## {decision["emoji"]} {decision["recommendation"]}

**Eligibility:** {decision["eligibility_label"]}
"""

    scores = f"""
| Component | Score |
|---|---:|
| Skills | **{skill_score}%** |
| Experience | **{experience_score}%** |
| Education | **{education_score}%** |
| Responsibilities | **{responsibility_score}%** |
"""

    details = "## 🧠 Skills\n\n"

    details += "### Required Skills\n\n"

    for skill in required_matched:
        details += f"✓ **{skill}**\n\n"

    for skill in required_missing:
        details += f"✗ **{skill}** — Missing\n\n"

    details += "### Preferred Skills\n\n"

    for skill in preferred_matched:
        details += f"✓ **{skill}**\n\n"

    for skill in preferred_missing:
        details += f"⚠ **{skill}** — Missing\n\n"

    details += "## 💪 Strengths\n\n"

    for item in decision.get("strengths", []):
        details += f"✓ {item}\n\n"

    details += "## ⚠️ Gaps\n\n"

    gaps = decision.get("gaps", [])

    if gaps:
        for item in gaps:
            details += f"⚠ {item}\n\n"
    else:
        details += "No major gaps detected.\n\n"

    details += "## 💼 Experience\n\n"
    details += (
        f"Candidate Experience: **{analysis['candidate_experience']} years**\n\n"
        f"Required Experience: **{analysis['required_experience']} years**\n\n"
        f"Experience Match: **{experience_score}%**\n\n"
    )

    education = analysis["candidate_education"]

    details += "## 🎓 Education\n\n"
    details += (
        f"Degrees: **{', '.join(education.get('degrees', [])) or 'Not detected'}**\n\n"
        f"Fields: **{', '.join(education.get('fields', [])) or 'Not detected'}**\n\n"
        f"Education Match: **{education_score}%**\n\n"
    )

    details += "## 🎯 Responsibilities\n\n"
    details += (
        f"Responsibility Match: **{responsibility_score}%**\n\n"
    )

    details += "## 🔎 Why?\n\n"
    details += decision.get(
        "explanation",
        "No explanation available."
    )

    return (
        "✅ Analysis completed successfully.",
        summary,
        scores + "\n" + details,
        analysis,
        cv_text_input
    )


print("Analysis callback ready.")


V2 — Structured Candidate Profile
Structured candidate profile generated successfully.
Skills, experience, education and project information identified.


In [ ]:
# ============================================================
# 7. CHATBOT CALLBACK
# ============================================================

def chat_with_analysis(message, history, analysis):

    if not message or not message.strip():
        return history or []

    history = history or []

    if analysis is None:

        answer = (
            "Please upload your CV, paste the Job Description, "
            "and click **Analyze Job** first."
        )

    elif "generate_chat_response_v2" not in globals():

        answer = (
            "The V2.22 chatbot engine is not loaded. "
            "Please run the V2.22 engine section."
        )

    else:

        try:
            answer = generate_chat_response_v2(
                question=message,
                decision=analysis["decision"],
                skill_result=analysis["skill_result"],
                experience_score=analysis["experience_result"]["experience_match"],
                candidate_experience=analysis["candidate_experience"],
                required_experience=analysis["required_experience"],
                education_result=analysis["education_result"],
                responsibility_result=analysis["responsibility_result"]
            )

        except Exception as e:

            answer = (
                "⚠️ Could not generate the answer.\n\n"
                f"Error: {e}"
            )

    # Compatible with the older Gradio Chatbot API.
    history.append([message, answer])

    return history


def clear_chat():
    return []


def clear_analysis():
    return (
        None,
        None,
        "",
        "",
        "",
        []
    )


print("Chatbot callbacks ready.")


V2 — Job Description Analysis
Job requirements extracted successfully.
Required skills and role requirements identified.


In [ ]:
# ============================================================
# 8. BUILD LATEST CHATBOT UI
# ============================================================

with gr.Blocks(
    title="AI Job Analyst V2"
) as demo:

    gr.Markdown(
        """
# 🤖 AI Job Analyst V2

### Evidence-grounded CV → Job Description analysis

**Upload your CV as PDF** and **paste the Job Description as text**.
"""
    )

    # --------------------------------------------------------
    # INPUTS
    # --------------------------------------------------------

    gr.Markdown("## 📥 Input")

    with gr.Row():

        with gr.Column():

            gr.Markdown("### 📄 Your CV")

            cv_upload = gr.File(
                label="Upload CV PDF",
                file_types=[".pdf"],
                type="filepath"
            )

        with gr.Column():

            gr.Markdown("### 📋 Job Description")

            jd_input = gr.Textbox(
                label="Paste Job Description",
                placeholder="Paste the complete Job Description here...",
                lines=15
            )

    analyze_button = gr.Button(
        "🔍 Analyze Job",
        variant="primary",
        size="lg"
    )

    status = gr.Markdown()

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    gr.Markdown("---")
    gr.Markdown("## 📊 Analysis Results")

    summary_output = gr.Markdown()

    scores_output = gr.Markdown()

    details_output = gr.Markdown()

    # --------------------------------------------------------
    # STATE
    # --------------------------------------------------------

    analysis_state = gr.State(None)

    cv_text_state = gr.State("")

    # --------------------------------------------------------
    # CHAT
    # --------------------------------------------------------

    gr.Markdown("---")
    gr.Markdown("## 💬 Ask AI Job Analyst")

    chatbot = gr.Chatbot(
        height=450
    )

    with gr.Row():

        chat_input = gr.Textbox(
            label="Your question",
            placeholder="Ask: Am I a good fit?",
            scale=8
        )

        chat_button = gr.Button(
            "Ask AI",
            variant="primary",
            scale=2
        )

    gr.Examples(
        examples=[
            ["Am I a good fit?"],
            ["What are my strengths?"],
            ["What are my gaps?"],
            ["Why am I borderline?"],
            ["Should I apply?"],
            ["How much experience do I have?"],
            ["What skills am I missing?"]
        ],
        inputs=chat_input
    )

    clear_chat_button = gr.Button(
        "🗑️ Clear Chat"
    )

    # --------------------------------------------------------
    # ANALYZE
    # --------------------------------------------------------

    analyze_button.click(
        fn=analyze_cv_pdf_and_jd_text,
        inputs=[
            cv_upload,
            jd_input
        ],
        outputs=[
            status,
            summary_output,
            scores_output,
            analysis_state,
            cv_text_state
        ]
    )

    # --------------------------------------------------------
    # CHAT
    # --------------------------------------------------------

    chat_button.click(
        fn=chat_with_analysis,
        inputs=[
            chat_input,
            chatbot,
            analysis_state
        ],
        outputs=[
            chatbot
        ]
    )

    chat_input.submit(
        fn=chat_with_analysis,
        inputs=[
            chat_input,
            chatbot,
            analysis_state
        ],
        outputs=[
            chatbot
        ]
    )

    # --------------------------------------------------------
    # CLEAR CHAT
    # --------------------------------------------------------

    clear_chat_button.click(
        fn=clear_chat,
        outputs=[
            chatbot
        ]
    )


print("Gradio UI created.")


V2 — CV–JD Matching
CV–Job Description matching completed successfully.
Evidence-based matching stage completed.


In [ ]:
# 9. LAUNCH
# For local use, run:
# demo.launch()

print("UI is ready. Uncomment demo.launch() to start the Gradio app.")


V2 — Chatbot
AI Job Analyst chatbot is ready.
Ask questions about the candidate profile, job requirements and match results.


## V2 Demonstration Output

This GitHub-ready version includes saved demonstration outputs showing the main stages of the AI Job Analyst workflow: CV extraction, candidate profiling, job-description analysis, CV–JD matching, and chatbot readiness.
